# Manufacturing AI Agent — RAG 테스트 (v2)

LangGraph 멀티에이전트에서 **RAG 파트만** 분리해 테스트하기 위한 노트북.
전체 에이전트(그래프 / 메모리 / 게이트 / 서브에이전트) 코드는 제거했고, RAG 구현·검증에 필요한 셀만 남겼다.

- 실행 순서: **0. 환경 → 1. LLM 어댑터 → 2. 계약(Pydantic) → 4. ChromaDB RAG**
- 사전 준비: `01_embed_documents_chroma.ipynb` 로 `document/` 를 ChromaDB에 임베딩해 둘 것.
- 핵심 진입점: `run_rag_pipeline(user_query, prediction)` → `EvidenceBundle` 반환.


## 0. 설치 & 환경

최초 1회만 실행. 이미 설치돼 있으면 건너뛴다. (uv 권장)


In [1]:
# 최초 1회만 실행 — 주석 해제 후 사용
# !uv pip install langgraph langgraph-checkpoint-sqlite langchain-core chromadb
# (선택) 실제 OpenAI LLM + 임베딩 사용 시 (langchain-openai가 openai 패키지를 함께 설치):
# !uv pip install langchain-openai openai
# (선택) 그래프 시각화:
# !uv pip install grandalf

print("설치 셀: 필요 시 위 주석을 해제해 실행하세요.")

설치 셀: 필요 시 위 주석을 해제해 실행하세요.


In [2]:
from __future__ import annotations

import os
import re
import json
from typing import Any, Optional, Literal

# pydantic (계약 스키마용)
from pydantic import BaseModel, Field

print("기본 import 완료 (RAG 테스트용)")


기본 import 완료 (RAG 테스트용)


## 1. 설정 & LLM 어댑터

`call_llm(system, user)` 하나로 통일한다.
- `langchain-openai` + `OPENAI_API_KEY` 가 있으면 실제 OpenAI 호출
- 없으면 결정론적 **StubLLM** 으로 폴백 → 오프라인에서도 노트북이 끝까지 실행됨


In [3]:
# ===================== 환경설정 (.env 로드) =====================
# API 키는 프로젝트 루트의 .env 파일에서 읽습니다. (.env.example 참고)
# 키를 이 노트북에 직접 적지 마세요 — .env 파일에만 저장합니다 (git에 커밋되지 않음).
# 실행 순서: 먼저 01_embed_documents_chroma.ipynb 를 실행한 뒤 이 노트북을 실행합니다.
#   .env 예시:  OPENAI_API_KEY=sk-proj-XXXXXXXX...

def load_dotenv(path: str = ".env") -> bool:
    if not os.path.exists(path):
        return False
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key and key not in os.environ:
                os.environ[key] = value
    return True

_ENV_PATH = ".env"
_ENV_EXISTS = os.path.exists(_ENV_PATH)
_ENV_LOADED = load_dotenv(_ENV_PATH)

# LangSmith tracing/upload 설정 (.env에서 LANGSMITH_*를 읽음)
LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")
LANGSMITH_TRACING = os.environ.get("LANGSMITH_TRACING", "true" if LANGSMITH_API_KEY else "false")
LANGSMITH_PROJECT = os.environ.get("LANGSMITH_PROJECT", "manufacturing-agent")
LANGSMITH_ENDPOINT = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")

os.environ["LANGSMITH_TRACING"] = LANGSMITH_TRACING
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT
if LANGSMITH_API_KEY:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

# LangChain/LangGraph 쪽 호환 환경변수도 같이 맞춘다.
os.environ["LANGCHAIN_TRACING_V2"] = LANGSMITH_TRACING
os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT
if LANGSMITH_API_KEY:
    os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY
# =========================================================

# 설정값
DEFAULT_MODEL = os.environ.get("OPENAI_CHAT_MODEL", "gpt-4o")               # 채팅 모델. 비용 민감 시 "gpt-4o-mini"
EMBED_MODEL = os.environ.get("OPENAI_EMBED_MODEL", "text-embedding-3-small") # 임베딩 모델. 고품질은 "text-embedding-3-large"
DATA_DIR = "agent_data"
os.makedirs(DATA_DIR, exist_ok=True)

LONGTERM_DB = os.path.join(DATA_DIR, "longterm_memory.sqlite")   # 장기 메모리 (대화/실행 이력)
CHECKPOINT_DB = os.path.join(DATA_DIR, "checkpoints.sqlite")     # 장기 체크포인터(SqliteSaver)
CHROMA_DIR = os.path.join(DATA_DIR, "chroma")                    # 벡터 스토어

_HAS_KEY = bool(os.environ.get("OPENAI_API_KEY"))
print(".env file:", "OK" if _ENV_EXISTS else "MISSING")
print(".env loaded:", "OK" if _ENV_LOADED else "SKIPPED")
print("OpenAI API key:", "OK" if _HAS_KEY else "MISSING")
print("Chat model:", DEFAULT_MODEL)
print("Embedding model:", EMBED_MODEL)

_LANGSMITH_ENABLED = LANGSMITH_TRACING.lower() in {"1", "true", "yes", "on"}
_LANGSMITH_HAS_KEY = bool(os.environ.get("LANGSMITH_API_KEY"))
print("LangSmith tracing:", "OK" if _LANGSMITH_ENABLED else "OFF")
print("LangSmith API key:", "OK" if _LANGSMITH_HAS_KEY else "MISSING")
print("LangSmith project:", LANGSMITH_PROJECT)
print("LangSmith endpoint:", LANGSMITH_ENDPOINT)

if _LANGSMITH_ENABLED and _LANGSMITH_HAS_KEY:
    try:
        from langsmith import Client
        _ls_client = Client(api_url=LANGSMITH_ENDPOINT, api_key=LANGSMITH_API_KEY)
        next(_ls_client.list_projects(limit=1), None)
        print("LangSmith upload check: OK")
    except Exception as e:
        print("LangSmith upload check: FAILED", e)
else:
    print("LangSmith upload check: SKIPPED")

_llm_client = None
_USE_REAL_LLM = False
try:
    if _HAS_KEY:
        from langchain_openai import ChatOpenAI
        _llm_client = ChatOpenAI(model=DEFAULT_MODEL, temperature=0, max_tokens=1024)
        _USE_REAL_LLM = True
except Exception as e:
    print("실제 LLM 비활성 (StubLLM 사용):", e)


def call_llm(system: str, user: str) -> str:
    """system+user 프롬프트 → 텍스트 응답. 미설치 시 StubLLM 폴백."""
    if _USE_REAL_LLM and _llm_client is not None:
        msg = _llm_client.invoke([("system", system), ("human", user)])
        return msg.content if isinstance(msg.content, str) else str(msg.content)
    return _stub_llm(system, user)


def _stub_llm(system: str, user: str) -> str:
    """결정론적 폴백: 입력을 요약해 자연어처럼 돌려준다(테스트/오프라인용)."""
    head = user.strip().splitlines()[0] if user.strip() else ""
    return f"[stub-llm 요약] {head[:160]}"


print("LLM 모드:", "REAL(" + DEFAULT_MODEL + ")" if _USE_REAL_LLM else "STUB")

.env file: OK
.env loaded: OK
OpenAI API key: OK
Chat model: gpt-4o-mini
Embedding model: text-embedding-3-small
LangSmith tracing: OK
LangSmith API key: OK
LangSmith project: manufacturing-agent
LangSmith endpoint: https://api.smith.langchain.com
LangSmith upload check: OK
LLM 모드: REAL(gpt-4o-mini)


## 2. `contracts/` — 데이터 계약 (Pydantic 스키마)

README 12장. Agent·Gate·Node가 주고받는 구조를 명확한 이름으로 정의한다.
`Artifact` 대신 `PredictionResult` / `EvidenceBundle` / `SafetyDecision` / `FinalAnswer` 등을 쓴다.


In [35]:
# ---------- contracts/context.py ----------
class ConversationTurn(BaseModel):
    role: str
    content: str
    created_at: str

class MachineValue(BaseModel):
    name: str
    value: float | str
    unit: Optional[str] = None
    source: str                       # "current" | "previous"
    is_current: bool
    is_stale: bool = False

class ContextPacket(BaseModel):
    current_question: str
    recent_turns_summary: str = ""
    selected_machine_values: dict[str, MachineValue] = {}
    previous_prediction_result: Optional[PredictionResult] = None
    previous_prediction_summary: Optional[str] = None
    #previous_safety_summary: Optional[str] = None
    user_constraints: dict = {}
    context_warnings: list[str] = []

class AgentContextPacket(BaseModel):
    agent_name: str
    current_question: str
    selected_context: dict = {}
    prior_results: dict = {}

# ---------- contracts/results.py ----------
class PredictionResult(BaseModel):
    status: str
    prediction_label: Optional[str] = None        # "normal" | "failure" 
    failure_types: list[dict] = []                #  [{"failure_type": "OSF"}, {"failure_type": "TWF"}]
    cause_features: list[str] = []                # ["torque", "tool_wear"]
    missing_features: list[str] = []
    full_prediction_available: bool = False
    prediction: Optional[dict] = None             # 계산식 기반 원본 결과
    summary: str = ""

# class EvidenceBundle(BaseModel):
#     retrieval_profile: str     # "default" | "safety" | "manufacturing" | "prediction"
#     queries: list[str] = []    # List of queries used to retrieve evidence
#     documents: list[dict] = [] # List of retrieved documents
#     citations: list[dict] = [] # List of citations for the retrieved documents
#     evidence_summary: str = "" # Summary of the retrieved evidence

class EvidenceBundle(BaseModel):
    mode: str = ""                         # "A" | "B"
    retrieval_profile: str                 # "troubleshooting_rag" | "prediction_plus_rag"
    user_query: str = ""
    search_query: str = ""
    tags: list[str] = []
    doc_whitelist: Optional[list[str]] = None
    failure_types: list[str] = []
    failure_ko: list[str] = []

    queries: list[str] = []
    documents: list[dict] = []
    citations: list[dict] = []
    evidence_summary: str = ""
    possible_prediction_based_evidence_query: bool = False


# class SafetyDecision(BaseModel):
#     risk_level: str                   # none | low | medium | high | critical
#     blocked: bool = False
#     forbidden_actions: list[str] = []
#     required_safety_notes: list[str] = []
#     summary: str = ""

class FinalAnswer(BaseModel):
    answer: str
    citations: list[dict] = []
    warnings: list[str] = []
    missing_inputs: list[str] = []

# ---------- contracts/routing.py ----------
class InputFlags(BaseModel):
    possible_manufacturing_query: bool = False #-> manufacturing 관련 evidence 검색 가능성 -??
    possible_prediction_query: bool = False    #-> prediction 관련 evidence 검색 가능성
    possible_evidence_query: bool = False      #-> RAG evidence 검색 가능성
    #possible_safety_query: bool = False
    possible_prompt_injection: bool = False
    contains_sensor_values: bool = False
    blocked_by_raw_input: bool = False

class RouteDecision(BaseModel):
    next_node: str
    reason: str
    stop: bool = False

class GateReport(BaseModel):
    gate_name: str
    status: str
    route_hint: Optional[str] = None
    reason: str = ""
    details: dict = {}

class RunTrace(BaseModel):
    request_id: str
    events: list[dict] = []

print("contracts 정의 완료")

contracts 정의 완료


## 4. ChromaDB RAG 구성

이 노트북은 **2번 실행 노트북**이다. 이미 임베딩된 ChromaDB 컬렉션을 열고 `EvidenceAgent`가 검색만 수행한다.

문서 임베딩은 **1번 준비 노트북**인 `01_embed_documents_chroma.ipynb`에서 최초 1회 또는 문서 변경 시 실행한다.

ChromaDB를 고정 사용한다. 인메모리 키워드 fallback은 두지 않는다.


In [36]:
# ---------- 2) Evidence RAG 런타임: 임베딩된 ChromaDB 검색만 수행 ----------
import hashlib

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from chromadb.utils import embedding_functions

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
LOCAL_EMBED_DIM = 384


class LocalHashEmbeddingFunction(EmbeddingFunction[Documents]):
    """외부 모델 다운로드 없이 동작하는 로컬 임베딩 함수."""

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            vec = [0.0] * LOCAL_EMBED_DIM
            tokens = re.findall(r"[A-Za-z가-힣0-9_]+", text.lower())
            for token in tokens:
                digest = hashlib.sha256(token.encode("utf-8")).digest()
                idx = int.from_bytes(digest[:4], "little") % LOCAL_EMBED_DIM
                sign = 1.0 if digest[4] % 2 == 0 else -1.0
                vec[idx] += sign
            norm = sum(v * v for v in vec) ** 0.5 or 1.0
            vectors.append([v / norm for v in vec])
        return vectors


def build_embedding_function():
    """01_embed_documents_chroma.ipynb의 임베딩 함수와 동일해야 한다."""
    if _HAS_KEY:
        return embedding_functions.OpenAIEmbeddingFunction(
            api_key=os.environ["OPENAI_API_KEY"], model_name=EMBED_MODEL), "manufacturing_document_chunks_openai", f"OpenAI({EMBED_MODEL})"
    return LocalHashEmbeddingFunction(), "manufacturing_document_chunks_local_hash", f"LocalHash({LOCAL_EMBED_DIM})"


_embed_fn, _collection_name, _embed_label = build_embedding_function()
_chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    _chroma_collection = _chroma_client.get_collection(
        _collection_name, embedding_function=_embed_fn)
except Exception as e:
    raise RuntimeError(
        f"ChromaDB 컬렉션 '{_collection_name}'을 찾을 수 없습니다. "
        "먼저 01_embed_documents_chroma.ipynb를 실행해 document/를 임베딩하세요."
    ) from e

print(f"Evidence RAG ChromaDB 연결 완료: collection={_collection_name}, embedding={_embed_label}, chunks={_chroma_collection.count()}")


def vector_search(query: str, k: int = 3, type_filter: Optional[str] = None) -> list[dict]:
    """이미 임베딩된 ChromaDB 컬렉션에서 관련 문서 top-k 검색."""
    where = {"type": type_filter} if type_filter else None
    res = _chroma_collection.query(query_texts=[query], n_results=k, where=where)
    docs = res.get("documents", [[]])[0]
    ids = res.get("ids", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0] if res.get("distances") else [0.0] * len(docs)
    out = []
    for i, doc in enumerate(docs):
        meta = metas[i] or {}
        out.append({
            "id": ids[i],
            "text": doc,
            "type": meta.get("type"),
            "source": meta.get("source"),
            "chunk_index": meta.get("chunk_index"),
            "score": 1.0 - float(distances[i]),
        })
    return out


print("Evidence RAG vector_search 준비 완료")

Evidence RAG ChromaDB 연결 완료: collection=manufacturing_document_chunks_openai, embedding=OpenAI(text-embedding-3-small), chunks=213
Evidence RAG vector_search 준비 완료


### 매핑 레이어 정의

In [37]:
FAILURE_RAG_MAP = {
    "공구마모고장": {
        "query_tags": [
            "tool wear",
            "chatter",
            "surface finish",
            "cutting load",
            "tool condition",
            "tool vibration"
        ],
        "documents": [
            "Mill Chatter",
            "Mill Spindle"
        ]
    },

    "열방출실패": {
        "query_tags": [
            "overheating",
            "spindle temperature",
            "lubrication",
            "cooling",
            "air pressure",
            "thermal issue"
        ],
        "documents": [
            "Mill Spindle",
            "Vector Drive"
        ]
    },

    "과부하파손": {
        "query_tags": [
            "overload",
            "high torque",
            "cutting force",
            "spindle load",
            "chatter",
            "rpm",
            "feed speed"
        ],
        "documents": [
            "Mill Chatter",
            "Mill Spindle",
            "Vector Drive"
        ]
    },

    "전력실패": {
        "query_tags": [
            "vector drive",
            "DC bus",
            "input voltage",
            "regen",
            "electrical failure",
            "spindle motor",
            "drive alarm"
        ],
        "documents": [
            "Vector Drive NGC",
            "Vector Drive CHC"
        ]
    },

    "무작위오류": {
        "query_tags": [
            "unknown failure",
            "alarm code",
            "symptom clarification"
        ],
        "documents": []
    }
}

In [22]:
VARIABLE_TAG_MAP = {
    "기온": [
        "ambient temperature",
        "cooling",
        "overheating"
    ],
    "공정온도": [
        "process temperature",
        "spindle overheating",
        "thermal issue",
        "lubrication"
    ],
    "회전속도": [
        "rpm",
        "spindle speed",
        "chatter",
        "vibration"
    ],
    "토크": [
        "torque",
        "high load",
        "overload",
        "cutting force",
        "spindle load"
    ],
    "공구마모": [
        "tool wear",
        "tool condition",
        "surface finish",
        "cutting load",
        "chatter"
    ]
}

## 4-1 기본 RAG 파이프라인 (Haas 문서 전용)

섹션 4에서 만든 `vector_search`와 섹션 1의 `call_llm`을 **그대로 재사용**해,
**Haas 트러블슈팅 문서만** 대상으로 기본 RAG 흐름을 시연한다.

흐름: **검색(retrieve) → 컨텍스트 주입(augment) → 답변 생성(generate)**

- Haas 문서는 `doc_type()` 규칙상 모두 `type="troubleshooting"`으로 임베딩되어 있다.
- `type="troubleshooting"` 필터로 1차로 좁히고, `source`가 `haas/`로 시작하는지 2차로 확인한다.
- 전체 에이전트 그래프(5번 이후)와 **독립적으로** 동작하는 학습/검증용 셀이다.


In [23]:
# ---------- 4-1) Haas 전용 retriever + 컨텍스트 구성 ----------
HAAS_DOC_TYPE = "troubleshooting"   # doc_type(): haas/ 문서는 모두 troubleshooting
HAAS_SOURCE_PREFIX = "haas/"


def haas_retrieve(query: str, k: int = 3) -> list[dict]:
    """Haas 트러블슈팅 문서만 대상으로 top-k 검색한다."""
    # type 필터로 1차로 좁히고, source 경로로 2차 확인(다른 문서 혼입 방지)
    hits = vector_search(query, k=k, type_filter=HAAS_DOC_TYPE)
    return [h for h in hits if (h.get("source") or "").startswith(HAAS_SOURCE_PREFIX)]


def format_evidence(hits: list[dict]) -> str:
    """검색 결과를 [1], [2] … 인용 번호가 붙은 컨텍스트 블록으로 변환한다."""
    blocks = []
    for i, h in enumerate(hits, start=1):
        loc = f"{h['source']}#chunk{h.get('chunk_index')}"
        blocks.append(f"[{i}] ({loc}, score={h['score']:.3f})\n{h['text']}")
    return "\n\n".join(blocks)


# 빠른 점검: Haas 문서가 실제로 검색되는지 확인
_probe = haas_retrieve("mill spindle troubleshooting", k=3)
print("Haas 검색 결과 수:", len(_probe))
for h in _probe:
    print(f" - {h['source']} (score={h['score']:.3f})")


Haas 검색 결과 수: 3
 - haas/Mill Spindle - Troubleshooting Guide - TG0101.html (score=0.627)
 - haas/Mill Spindle - Troubleshooting Guide - TG0101.html (score=0.625)
 - haas/Mill Spindle - Troubleshooting Guide - TG0101.html (score=0.613)


In [24]:
# ---------- 4-2) 기본 RAG 답변 생성 (retrieve -> augment -> generate) ----------
HAAS_RAG_SYSTEM = (
    "너는 Haas CNC 장비 트러블슈팅 보조원이다. "
    "아래 제공된 근거(evidence)만 사용해 한국어로 답하라. "
    "근거에 없는 내용은 추측하지 말고 '문서에서 확인되지 않음'이라고 말하라. "
    "답변 문장 끝에는 사용한 근거 번호를 [1], [2] 형식으로 표기하라."
)


def haas_rag_answer(query: str, k: int = 3) -> dict:
    """Haas 문서 대상 기본 RAG: 검색 -> 컨텍스트 주입 -> LLM 답변."""
    hits = haas_retrieve(query, k=k)
    if not hits:
        return {"query": query, "answer": "관련 Haas 문서를 찾지 못했습니다.", "hits": [], "citations": []}

    context = format_evidence(hits)
    user_prompt = (
        f"[질문]\n{query}\n\n"
        f"[근거 문서]\n{context}\n\n"
        "[지시] 위 근거만 사용해 답하고, 사용한 근거 번호를 [n] 형식으로 표기하라."
    )
    answer = call_llm(HAAS_RAG_SYSTEM, user_prompt)

    citations = [
        {"n": i, "source": h["source"], "chunk_index": h.get("chunk_index"), "score": h["score"]}
        for i, h in enumerate(hits, start=1)
    ]
    return {"query": query, "answer": answer, "hits": hits, "citations": citations}


print("haas_rag_answer 준비 완료")


haas_rag_answer 준비 완료


## 4-2 구조화 RAG 파이프라인 (서비스형)

4-1의 기본 RAG를 4단계로 구조화하되, **6번 `services`와 동일한 규약**으로 산출한다.

```
(1) Query Builder ── prediction 없음(mode A) → query 그대로        (profile=troubleshooting_rag)
                  └─ prediction 있음(mode B) → 매핑 태그로 재작성   (profile=prediction_plus_rag)
(2) Retriever        → Chroma 검색(haas_retrieve), mode B는 문서 화이트리스트로 추가 제한
(3) Evidence Ranker  → 중복 제거 후 score Top-k
(4) Evidence Summarizer → 6번식 근거 요약(evidence_summary) + build_citations
```

**6번에 맞춘 점**
- 목적: 단독 데모가 아니라 **에이전트 그래프에 끼울 수 있는 검색 서비스 컴포넌트**(Haas 한정 목업).
- 입출력 타입: 입력은 `PredictionResult`(2번 계약), 출력은 `EvidenceBundle`(2번 계약).
- 요약: 전체 답변 대신 6번처럼 **2~3문장 근거 요약**(`evidence_summary`).
- 인용: 6번 `build_citations` 포맷 `{source_id, type, snippet, score}`.

`PredictionResult.failure_type`는 AI4I 코드(HDF/PWF/OSF/TWF), 매핑 테이블은 한글 키이므로
(1)에서 코드<->한글 브리지로 연결한다. 4번 고유의 **mode A/B 질의 재작성 + 문서 화이트리스트**는 유지한다.


In [40]:
# ---------- (1) Query Builder ----------
# 입력: user_query + Optional[PredictionResult]  (6번과 동일한 계약 타입)
# 예측 결과는 '고장 유형(여러 개 가능)'과 '원인 변수'만 사용한다. (확률/score/confidence 미사용)
# - mode A: 고장 유형 없음 -> query 그대로,                       profile = "troubleshooting_rag"
# - mode B: 고장 유형 있음 -> 모든 유형의 매핑 태그로 재작성 + 화이트리스트, profile = "prediction_plus_rag"

# PredictionResult.failure_types의 AI4I 코드 <-> 4번 매핑 테이블(한글 키) 브리지
FAILURE_CODE_TO_KO = {
    "TWF": "공구마모고장", "HDF": "열방출실패",
    "OSF": "과부하파손", "PWF": "전력실패", "RNF": "무작위오류",
}
FEATURE_TO_KO = {
    "air_temperature": "기온", "process_temperature": "공정온도",
    "rotational_speed": "회전속도", "torque": "토크", "tool_wear": "공구마모",
}


def build_query(user_query: str, prediction: Optional[PredictionResult] = None) -> dict:
    """(1) 검색 계획 수립. 고장 유형 유무로 mode A/B 분기 (확률/score 미사용)."""
    risks = prediction.failure_types if prediction else []
    if not risks:
        return {"mode": "A", "profile": "troubleshooting_rag", "user_query": user_query,
                "search_query": user_query, "tags": [], "doc_whitelist": None,
                "failure_types": [], "failure_ko": []}

    # 도출된 고장 유형을 '모두' 반영한다 (확률로 1개만 고르지 않는다).
    failure_types, failure_ko, tags, docs = [], [], [], []
    for r in risks:
        code = r.get("failure_type")
        failure_types.append(code)
        ko = FAILURE_CODE_TO_KO.get(code)
        failure_ko.append(ko)
        fmap = FAILURE_RAG_MAP.get(ko, {})
        tags.extend(fmap.get("query_tags", []))
        docs.extend(fmap.get("documents", []))

    # 원인 변수 -> 태그 (cause_features를 원인 변수로 사용)
    for feat in (prediction.cause_features or []):
        tags.extend(VARIABLE_TAG_MAP.get(FEATURE_TO_KO.get(feat, ""), []))

    tags = list(dict.fromkeys(tags))   # 순서 유지 + 중복 제거
    docs = list(dict.fromkeys(docs))
    return {"mode": "B", "profile": "prediction_plus_rag", "user_query": user_query,
            "search_query": " ".join([user_query, *tags]).strip(), "tags": tags,
            "doc_whitelist": docs or None,
            "failure_types": failure_types, "failure_ko": failure_ko}


print("(1) build_query 준비 완료")


(1) build_query 준비 완료


In [41]:
# ---------- (2) Retriever (Chroma Search) ----------
def _doc_name_matches(source: str, doc_name: str) -> bool:
    """매핑의 친숙한 문서명('Vector Drive NGC')이 실제 source 경로에 모두 포함되는지."""
    s = (source or "").lower()
    return all(tok.lower() in s for tok in doc_name.split())


def retrieve_stage(plan: dict, k: int = 8) -> list[dict]:
    """(2) Haas 컬렉션 검색. mode B면 문서 화이트리스트로 추가 제한."""
    hits = haas_retrieve(plan["search_query"], k=k)   # 4-1 재사용 (type=troubleshooting + haas/)
    whitelist = plan.get("doc_whitelist")
    if whitelist:
        hits = [h for h in hits
                if any(_doc_name_matches(h.get("source", ""), name) for name in whitelist)]
    return hits


print("(2) retrieve_stage 준비 완료")


(2) retrieve_stage 준비 완료


In [42]:
# ---------- (3) Evidence Ranker (Top-k) ----------
def rank_evidence(hits: list[dict], top_k: int = 3) -> list[dict]:
    """(3) (source, chunk) 중복 제거 후 score 내림차순 Top-k."""
    seen, ranked = set(), []
    for h in sorted(hits, key=lambda x: x.get("score", 0.0), reverse=True):
        key = (h.get("source"), h.get("chunk_index"))
        if key in seen:
            continue
        seen.add(key)
        ranked.append(h)
        if len(ranked) >= top_k:
            break
    return ranked


print("(3) rank_evidence 준비 완료")


(3) rank_evidence 준비 완료


In [43]:
# ---------- (4) Evidence Summarizer (6번 규약: evidence_summary + build_citations) ----------
# 인용 포맷은 6번 services/citation_service.py와 동일하게 맞춘다.
def build_citations(docs: list[dict]) -> list[dict]:
    return [{"source_id": d["id"], "type": d.get("type"),
             "snippet": d["text"][:120], "score": round(float(d.get("score", 0)), 3)}
            for d in docs]


# 6번 evidence_agent와 동일한 '근거 수집가' 스타일 2~3문장 요약
EVIDENCE_SUMMARY_SYSTEM = (
    "너는 근거 수집가다. 검색 문서를 바탕으로 핵심 근거를 2~3문장으로 요약하라. "
    "문서에 없는 내용은 만들지 마라."
)


def summarize_evidence(user_query: str, ranked: list[dict]) -> str:
    """(4) Top-k 근거를 evidence_summary 문자열로 요약(6번식)."""
    if not ranked:
        return "관련 Haas 근거를 찾지 못했습니다."
    return call_llm(
        EVIDENCE_SUMMARY_SYSTEM,
        f"질문:{user_query}\n문서:{json.dumps([d['text'] for d in ranked], ensure_ascii=False)}")


print("(4) summarize_evidence / build_citations 준비 완료")


(4) summarize_evidence / build_citations 준비 완료


In [44]:
# ---------- (1)->(2)->(3)->(4) 연결: EvidenceBundle 산출 (6번 규약) ----------
def run_rag_pipeline(user_query: str, prediction: Optional[PredictionResult] = None,
                     retrieve_k: int = 8, top_k: int = 3) -> EvidenceBundle:
    """4단계를 연결해 6번과 동일한 EvidenceBundle을 돌려준다. (Haas 한정 목업)"""
    plan = build_query(user_query, prediction)        # (1)
    hits = retrieve_stage(plan, k=retrieve_k)          # (2)
    ranked = rank_evidence(hits, top_k=top_k)          # (3)
    summary = summarize_evidence(user_query, ranked)   # (4)
    return EvidenceBundle(
        retrieval_profile=plan["profile"],
        queries=[plan["search_query"]],
        documents=ranked,
        citations=build_citations(ranked),
        evidence_summary=summary,
    )


def print_evidence_bundle(bundle: EvidenceBundle) -> None:
    mode = "B" if bundle.retrieval_profile == "prediction_plus_rag" else "A"
    print(f"[mode {mode}] retrieval_profile={bundle.retrieval_profile}")
    print("queries:", bundle.queries)
    print(f"documents(top-k)={len(bundle.documents)}")
    print("\n[evidence_summary]\n", bundle.evidence_summary)
    print("\n[citations]")
    for c in bundle.citations:
        print(f"  - {c['source_id']} ({c['type']}, score={c['score']}) {c['snippet'][:50]}...")


print("run_rag_pipeline 준비 완료")


run_rag_pipeline 준비 완료


In [45]:
# ---------- 데모 시나리오 1) mode A: 단순 질의 (예측 없음) ----------
_bundle_a = run_rag_pipeline("밀링 채터(chatter)가 발생하는 원인과 해결 방법은?")
print_evidence_bundle(_bundle_a)


[mode A] retrieval_profile=troubleshooting_rag
queries: ['밀링 채터(chatter)가 발생하는 원인과 해결 방법은?']
documents(top-k)=3

[evidence_summary]
 밀링 채터(chatter)는 주로 너무 많은 플루트가 절삭에 참여하거나, 절삭 경로의 변화로 인해 발생하는 힘의 급증으로 인해 발생한다. 해결 방법으로는 플루트 수를 줄이거나 절삭 깊이 및 폭을 감소시키고, 일정한 절삭 힘을 유지하는 도구 경로를 사용하는 것이 있다. 또한, 도구의 마모 상태를 점검하고, 적절한 도구 직경을 사용하는 것도 중요하다.

[citations]
  - b0088134a96305e1 (troubleshooting, score=0.425) n too many flutes are engaged in the cut [1], incr...
  - 242f2c9764ba8c39 (troubleshooting, score=0.374) s
Welcome,
Haas Tooling
MyHaas/HaasConnect
Sign In...
  - 63c489bd31879be5 (troubleshooting, score=0.348) or clearance all the way around with a piece of 0....


In [46]:
# ---------- 데모 시나리오 2) mode B: PredictionResult 기반 질의 재작성 ----------
# 예측 모듈이 '고장 유형(여러 개)'과 '원인 변수'만 넘긴다고 가정한다. (확률/score 없음)
# class PredictionResult(BaseModel):
#     status: str
#     prediction_label: Optional[str] = None        # "normal" | "failure" 
#     failure_types: list[dict] = []                #  [{"failure_type": "OSF"}, {"failure_type": "TWF"}]
#     cause_features: list[str] = []                # ["torque", "tool_wear"]
#     missing_features: list[str] = []
#     full_prediction_available: bool = False
#     prediction: Optional[dict] = None             # 계산식 기반 원본 결과
#     summary: str = ""

_pred = PredictionResult(
    status="PARTIAL",
    cause_features=["torque", "tool_wear", "rotational_speed"],   # 원인 변수
    failure_types=[
        {"failure_type": "OSF"},   # 과부하파손
        {"failure_type": "TWF"},   # 공구마모고장
    ],
)
_bundle_b = run_rag_pipeline("스핀들 부하가 높을 때 점검해야 할 항목은?", prediction=_pred)
print_evidence_bundle(_bundle_b)


[mode B] retrieval_profile=prediction_plus_rag
queries: ['스핀들 부하가 높을 때 점검해야 할 항목은? overload high torque cutting force spindle load chatter rpm feed speed tool wear surface finish cutting load tool condition tool vibration torque high load spindle speed vibration']
documents(top-k)=3

[evidence_summary]
 스핀들 부하가 높을 때 점검해야 할 항목으로는 스핀들 윤활 시스템의 작동 여부, 도구 홀더 및 스핀들 테이퍼의 청소 및 손상 여부, 그리고 베어링 상태를 확인하는 것이 중요하다. 또한, 도구의 길이가 너무 길거나 불균형이 있을 경우에도 문제가 발생할 수 있으므로, 도구의 길이를 줄이거나 균형을 맞추는 조치를 취해야 한다.

[citations]
  - 084cdf527f87df34 (troubleshooting, score=0.556) the spindle taper before it is released.
A common ...
  - 59310f70c706b4e6 (troubleshooting, score=0.555) Maintain and inspect tool holders, and spindle tap...
  - 607ec1b2d3c3edda (troubleshooting, score=0.553) ication system is not functioning correctly.
Inspe...
